### Drift Monitoring and Retraining

A model's performance at deployment time is not permanent, the world the model was trained on keeps changing. Covers the types of drift, how to detect them (PSI, KS-test), what to monitor in production, and how to decide when to retrain.

#### 0. Types of drift

Data drift (covariate shift): the distribution of INPUT features changes, but the true relationship between features and target stays the same. Example: a fraud model trained when `amount_lost` averaged $2,000 now sees transactions averaging $8,000, because inflation or a new user segment, the fraud mechanics themselves have not changed, but the model's training distribution no longer matches production.

Concept drift: the relationship between features and target itself changes, the same input now means something different. Example: `payment_method_requested=gift_card` used to be a strong fraud signal, but if legitimate businesses started commonly requesting gift card payments for unrelated reasons, that same feature value now means something different than it did at training time.

Label drift: the distribution of the TARGET itself shifts, independent of feature drift, e.g. a genuinely new fraud typology emerges that did not exist in training data at all (directly relevant to the fraud-theme-detection project, trained on a fixed list of 10 known typologies, a new scam pattern would not even have a label to be assigned to).

#### 1. PSI (Population Stability Index), worked by hand

Formula: PSI = sum over bins of (actual% - expected%) * ln(actual% / expected%), where "expected" is the reference (training) distribution and "actual" is the current (production) distribution.

Worked example, feature = amount_lost, binned into 3 buckets:
```
                low    medium   high
reference:      50%    30%      20%
current:        30%    30%      40%

PSI = (0.30-0.50)*ln(0.30/0.50) + (0.30-0.30)*ln(0.30/0.30) + (0.40-0.20)*ln(0.40/0.20)
    = (-0.20)*ln(0.6) + 0*ln(1) + (0.20)*ln(2.0)
    = (-0.20)*(-0.511) + 0 + (0.20)*(0.693)
    = 0.102 + 0 + 0.139
    = 0.241
```
Standard interpretation thresholds: PSI < 0.1 = no significant shift, 0.1-0.25 = moderate shift, worth investigating, PSI > 0.25 = significant shift, retraining likely needed. 0.241 lands in "moderate, investigate" territory, right at the edge of the retrain threshold, exactly the kind of borderline case a monitoring dashboard should flag for a human to look at rather than silently ignore or silently auto-retrain on.

In [ ]:
import numpy as np

reference = np.array([0.50, 0.30, 0.20])
current = np.array([0.30, 0.30, 0.40])

psi_per_bin = (current - reference) * np.log(current / reference)
psi = np.sum(psi_per_bin)

print("PSI per bin:", psi_per_bin.round(3))
print("total PSI:", psi.round(3))
print("interpretation:", "stable" if psi < 0.1 else ("investigate" if psi < 0.25 else "retrain"))

#### 2. KS-test (Kolmogorov-Smirnov), worked by hand

The KS statistic is the maximum vertical distance between two empirical CDFs (cumulative distribution functions), a distribution-free way to compare a reference sample and a current sample without binning.

Worked example, tiny samples: reference=[1,2,3,4,5], current=[3,4,5,6,7] (a clean shift upward).
```
value:        1    2    3    4    5    6    7
F_reference:  0.2  0.4  0.6  0.8  1.0  1.0  1.0
F_current:    0.0  0.0  0.2  0.4  0.6  0.8  1.0
difference:   0.2  0.4  0.4  0.4  0.4  0.2  0.0
```
Maximum difference (the KS statistic, D) = 0.4, occurring at several points (values 2 through 5). A larger D means the two distributions differ more; a p-value (from the KS distribution given both sample sizes) then says whether that D is large enough to be statistically significant rather than just sampling noise, not covered in detail here, the intuition (max gap between two CDFs) is the part worth internalizing.

In [ ]:
from scipy import stats

reference_sample = [1, 2, 3, 4, 5]
current_sample = [3, 4, 5, 6, 7]

ks_stat, p_value = stats.ks_2samp(reference_sample, current_sample)
print("KS statistic:", ks_stat)
print("p-value:", p_value)

#### 3. What to actually monitor in production

- Feature distributions: PSI/KS-test per feature, catches data drift before it shows up as a performance drop.
- Prediction distributions: is the model's OUTPUT distribution shifting (e.g. suddenly predicting a much higher rate of one class), even without knowing the true labels yet, a real early-warning signal.
- Performance metrics, once labels arrive: precision/recall/F1 (see `eval-metrics.ipynb`) computed on a rolling window of recent labeled data, the direct measure, but delayed by however long it takes labels to actually arrive (in fraud, labels come from chargebacks/confirmed reports, which can lag the original transaction by weeks).
- Calibration drift (see `eval-metrics.ipynb`'s calibration section): even if ranking (AUC) stays fine, the model's probabilities can become miscalibrated over time as the underlying rates shift, silently breaking any downstream threshold-based decision.

#### 4. Retraining triggers

Threshold-based: retrain when PSI/KS crosses a set level on a monitored feature, or when a performance metric drops below a set floor. Reactive but precise, ties retraining directly to evidence of actual drift.

Scheduled: retrain on a fixed calendar (weekly, monthly) regardless of measured drift. Simple, predictable, but wastes compute retraining when nothing has changed, and can still miss a sudden drift event that happens mid-cycle.

Performance-based: retrain when live performance metrics (once labels catch up) fall below a floor, the most direct trigger, but the most delayed, by definition it only fires after the model has already been degraded in production for however long labels take to arrive.

Practical pattern: combine them, PSI/KS as an early warning (fast, no label lag) that triggers investigation, performance metrics as the final confirmation before committing to a full retrain, since drift in a feature does not always translate into an actual performance drop, some drifted features may not matter much to the model's decision boundary.